# World Weather Online Historical Weather Ingestion

Pull historical weather data for NFL stadiums to build weather similarity models.

**Features:**
- Free tier: 500 API calls/day
- Historical weather data (past games)
- Current weather and 14-day forecast (paid)
- Hourly historical data
- Wind speed, gusts, temperature, precipitation

**Use Case: Historical Weather Similarity Search**
- Find past games with similar weather conditions
- Analyze player performance in comparable weather
- Build weather-adjusted projections based on historical patterns
- Example: "Find all games with 20mph winds, 35°F, similar to today's forecast"

**Resources:**
- Website: https://www.worldweatheronline.com
- API Docs: https://www.worldweatheronline.com/developer/api/
- Free tier: 500 calls/day
- Sign up: https://www.worldweatheronline.com/developer/api/pricing.aspx

In [0]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ============================================================================
# CONFIGURATION MODE
# ============================================================================
# WorldWeatherOnline Use Case: PRE-GAME HOURLY WEATHER UPDATES
# Run this notebook Friday/Saturday/Sunday morning before games
# Get hour-by-hour weather for precise game-time conditions
# ============================================================================

PREGAME_HOURS = 48  # Fetch games in next 48 hours (adjustable)

# WorldWeatherOnline configuration
BASE_URL = "https://api.worldweatheronline.com/premium/v1"

# Get API key from Databricks Secrets
try:
    API_KEY = dbutils.secrets.get(scope="fantasai", key="wwo_api_key")
    print("✅ API key loaded from secrets\n")
except:
    # Use direct API key for this run
    API_KEY = "c4c1bc19d9194be19f8133911263105"
    print("✅ API key loaded from configuration\n")
    print("💡 Store in secrets: Secrets UI → scope: fantasai → key: wwo_api_key")
    print()

print("⏱️ PRE-GAME HOURLY MODE: Fetch hour-by-hour weather for upcoming games")
print(f"   Looking for games in next {PREGAME_HOURS} hours")
print("   Using /past-weather.ashx endpoint (works for today/yesterday)")
print("   🎯 Perfect for last-minute lineup decisions based on evolving weather")
print("\n📌 Recommended Schedule:")
print("   Friday 8am: Run for Sunday games (get hourly trends)")
print("   Saturday 8am: Run for Sunday games (updated hourly forecast)")
print("   Sunday 10am: Run for 1pm/4pm games (final weather check)")
print(f"\n📊 Free tier: 500 calls/day (~16 games = ~32 calls for hourly data)")

In [0]:
# NFL stadium locations with historical franchise relocation support
# Same function as WeatherAPI.com notebook for consistency

# Current NFL stadiums (2020-2025)
NFL_STADIUMS_CURRENT = {
    'ARI': ('Glendale, AZ', 'State Farm Stadium', True),
    'ATL': ('Atlanta, GA', 'Mercedes-Benz Stadium', True),
    'BAL': ('Baltimore, MD', 'M&T Bank Stadium', False),
    'BUF': ('Orchard Park, NY', 'Highmark Stadium', False),
    'CAR': ('Charlotte, NC', 'Bank of America Stadium', False),
    'CHI': ('Chicago, IL', 'Soldier Field', False),
    'CIN': ('Cincinnati, OH', 'Paycor Stadium', False),
    'CLE': ('Cleveland, OH', 'Cleveland Browns Stadium', False),
    'DAL': ('Arlington, TX', 'AT&T Stadium', True),
    'DEN': ('Denver, CO', 'Empower Field at Mile High', False),
    'DET': ('Detroit, MI', 'Ford Field', True),
    'GB': ('Green Bay, WI', 'Lambeau Field', False),
    'HOU': ('Houston, TX', 'NRG Stadium', True),
    'IND': ('Indianapolis, IN', 'Lucas Oil Stadium', True),
    'JAX': ('Jacksonville, FL', 'TIAA Bank Field', False),
    'KC': ('Kansas City, MO', 'GEHA Field at Arrowhead Stadium', False),
    'LAC': ('Inglewood, CA', 'SoFi Stadium', False),
    'LAR': ('Inglewood, CA', 'SoFi Stadium', False),
    'LV': ('Las Vegas, NV', 'Allegiant Stadium', True),
    'MIA': ('Miami Gardens, FL', 'Hard Rock Stadium', False),
    'MIN': ('Minneapolis, MN', 'U.S. Bank Stadium', True),
    'NE': ('Foxborough, MA', 'Gillette Stadium', False),
    'NO': ('New Orleans, LA', 'Caesars Superdome', True),
    'NYG': ('East Rutherford, NJ', 'MetLife Stadium', False),
    'NYJ': ('East Rutherford, NJ', 'MetLife Stadium', False),
    'PHI': ('Philadelphia, PA', 'Lincoln Financial Field', False),
    'PIT': ('Pittsburgh, PA', 'Acrisure Stadium', False),
    'SEA': ('Seattle, WA', 'Lumen Field', False),
    'SF': ('Santa Clara, CA', "Levi's Stadium", False),
    'TB': ('Tampa, FL', 'Raymond James Stadium', False),
    'TEN': ('Nashville, TN', 'Nissan Stadium', False),
    'WAS': ('Landover, MD', 'FedExField', False)
}

NFL_STADIUMS_HISTORICAL = {
    'OAK': ('Oakland, CA', 'Oakland Coliseum', False),
    'SD': ('San Diego, CA', 'Qualcomm Stadium', False),
    'LAC_2017': ('Carson, CA', 'Dignity Health Sports Park', False),
    'LA': ('Los Angeles, CA', 'Los Angeles Memorial Coliseum', False),
    'LAR_2016': ('Los Angeles, CA', 'Los Angeles Memorial Coliseum', False),
}

def get_stadium_for_season(team_code, season):
    if team_code == 'OAK':
        return NFL_STADIUMS_HISTORICAL['OAK'] if season <= 2019 else NFL_STADIUMS_CURRENT['LV']
    if team_code == 'LV':
        return NFL_STADIUMS_CURRENT['LV'] if season >= 2020 else NFL_STADIUMS_HISTORICAL['OAK']
    if team_code == 'SD':
        return NFL_STADIUMS_HISTORICAL['SD']
    if team_code == 'LAC':
        if season == 2016:
            return NFL_STADIUMS_HISTORICAL['SD']
        elif season <= 2019:
            return NFL_STADIUMS_HISTORICAL['LAC_2017']
        else:
            return NFL_STADIUMS_CURRENT['LAC']
    if team_code in ('LA', 'LAR'):
        return NFL_STADIUMS_HISTORICAL['LAR_2016'] if season <= 2019 else NFL_STADIUMS_CURRENT['LAR']
    if team_code in NFL_STADIUMS_CURRENT:
        return NFL_STADIUMS_CURRENT[team_code]
    raise ValueError(f"Unknown team code: {team_code} for season {season}")

print(f"✅ Loaded {len(NFL_STADIUMS_CURRENT)} current NFL stadiums")
print(f"✅ Loaded {len(NFL_STADIUMS_HISTORICAL)} historical stadium mappings")

In [0]:
# Fetch upcoming games in the next 48 hours for pre-game hourly weather
from datetime import datetime, timedelta

print("="*80)
print("🏈 Fetching Upcoming NFL Games")
print("="*80)

# Calculate time window
now = datetime.now()
window_end = now + timedelta(hours=PREGAME_HOURS)

today_str = now.strftime('%Y-%m-%d')
window_end_str = window_end.strftime('%Y-%m-%d')

print(f"\n📅 Current time: {now.strftime('%Y-%m-%d %H:%M')}")
print(f"📅 Looking for games through: {window_end.strftime('%Y-%m-%d %H:%M')}")

# Fetch games in the next 48 hours
games_query = f"""
SELECT 
    event_id,
    season,
    week,
    date,
    home_team,
    away_team,
    venue,
    game_type
FROM main.fantasai.bronze_nfl_games
WHERE source = 'nflverse'
  AND game_type = 'REG'
  AND date >= '{today_str}'
  AND date <= '{window_end_str}'
ORDER BY date, home_team
"""

games_df = spark.sql(games_query).toPandas()

if len(games_df) > 0:
    print(f"\n✅ Found {len(games_df)} upcoming games")
    print(f"   Date range: {games_df['date'].min()} to {games_df['date'].max()}")
    print(f"\n🎯 Games to fetch hourly weather for:")
    display(games_df[['date', 'home_team', 'away_team', 'venue']])
else:
    print(f"\n😴 No games in next {PREGAME_HOURS} hours")
    print("\n📌 Check back:")
    print("   Friday for Sunday games")
    print("   Saturday for Sunday games")
    print("   Sunday morning for afternoon games")

In [0]:
# Fetch hour-by-hour weather for each upcoming game
import time

if API_KEY is None:
    print("❌ Cannot fetch weather - API key not configured")
elif len(games_df) == 0:
    print("⚠️ No upcoming games to fetch weather for")
else:
    print("\n" + "="*80)
    print("⏱️ Fetching Hourly Weather Data")
    print("="*80)
    
    hourly_weather_records = []
    api_calls = 0
    errors = 0
    
    for idx, game in games_df.iterrows():
        game_date = game['date']
        home_team = game['home_team']
        season = int(game['season'])
        
        # Get stadium location
        try:
            city, stadium, is_dome = get_stadium_for_season(home_team, season)
        except ValueError as e:
            print(f"  ⚠️ {e}")
            errors += 1
            continue
        
        print(f"\n🌦️ {game_date} - {home_team} at {city}")
        
        try:
            # WorldWeatherOnline past-weather endpoint (works for today and past dates)
            url = f"{BASE_URL}/past-weather.ashx"
            params = {
                'key': API_KEY,
                'q': city,
                'date': game_date,
                'tp': '1',  # 1-hour intervals
                'format': 'json'
            }
            
            response = requests.get(url, params=params, timeout=15)
            response.raise_for_status()
            api_calls += 1
            
            data = response.json()
            
            # Extract hourly weather
            weather_data = data.get('data', {}).get('weather', [{}])[0]
            hourly_data = weather_data.get('hourly', [])
            
            if not hourly_data:
                print(f"  ⚠️ No hourly data available for {game_date}")
                errors += 1
                continue
            
            print(f"  ✅ Fetched {len(hourly_data)} hourly records")
            
            # Parse each hour
            for hour in hourly_data:
                hour_int = int(hour.get('time', '0')) // 100  # Convert '1300' to 13
                
                hourly_record = {
                    'event_id': game['event_id'],
                    'season': season,
                    'week': int(game['week']),
                    'game_date': game_date,
                    'hour': hour_int,
                    'team': home_team,
                    'city': city,
                    'stadium': stadium,
                    'is_dome': is_dome,
                    # Temperature
                    'temp_f': float(hour.get('tempF', 0)),
                    'temp_c': float(hour.get('tempC', 0)),
                    'feels_like_f': float(hour.get('FeelsLikeF', 0)),
                    'feels_like_c': float(hour.get('FeelsLikeC', 0)),
                    # Wind (CRITICAL for pre-game decisions)
                    'wind_speed_mph': float(hour.get('windspeedMiles', 0)),
                    'wind_speed_kmph': float(hour.get('windspeedKmph', 0)),
                    'wind_gust_mph': float(hour.get('WindGustMiles', 0)),
                    'wind_gust_kmph': float(hour.get('WindGustKmph', 0)),
                    'wind_dir': hour.get('winddir16Point', ''),
                    'wind_degree': int(hour.get('winddirDegree', 0)),
                    # Precipitation
                    'precip_in': float(hour.get('precipInches', 0)),
                    'precip_mm': float(hour.get('precipMM', 0)),
                    'humidity': int(hour.get('humidity', 0)),
                    'chance_of_rain': int(hour.get('chanceofrain', 0)),
                    'chance_of_snow': int(hour.get('chanceofsnow', 0)),
                    # Conditions
                    'condition': hour.get('weatherDesc', [{}])[0].get('value', ''),
                    'condition_code': int(hour.get('weatherCode', 0)),
                    'cloud_cover': int(hour.get('cloudcover', 0)),
                    'visibility_miles': float(hour.get('visibilityMiles', 0)),
                    # Full JSON
                    'raw_data': json.dumps(hour)
                }
                
                hourly_weather_records.append(hourly_record)
            
            # Show key hours (10am, 1pm, 4pm, 8pm)
            key_hours = [10, 13, 16, 20]
            key_weather = [h for h in hourly_data if int(h.get('time', '0')) // 100 in key_hours]
            
            if key_weather:
                print(f"  🕑 Key game times:")
                for h in key_weather:
                    hour_int = int(h.get('time', '0')) // 100
                    temp = h.get('tempF')
                    wind = h.get('windspeedMiles')
                    gust = h.get('WindGustMiles')
                    condition = h.get('weatherDesc', [{}])[0].get('value', '')
                    print(f"       {hour_int}:00 - {temp}°F, Wind {wind}mph (gusts {gust}mph), {condition}")
            
            # Rate limiting
            time.sleep(0.2)  # 5 requests/second
            
        except requests.exceptions.HTTPError as e:
            print(f"  ❌ HTTP Error {e.response.status_code}")
            errors += 1
        except Exception as e:
            print(f"  ❌ Error: {str(e)[:80]}")
            errors += 1
    
    print("\n" + "="*80)
    print("📊 Hourly Weather Fetch Summary")
    print("="*80)
    print(f"  Games processed: {len(games_df)}")
    print(f"  Hourly records fetched: {len(hourly_weather_records)}")
    print(f"  API calls made: {api_calls}")
    print(f"  Errors: {errors}")
    
    if len(hourly_weather_records) > 0:
        hourly_weather_df = pd.DataFrame(hourly_weather_records)
        print(f"\n✅ Created hourly weather DataFrame")
        
        # Show wind trends
        print(f"\n🌬️ Wind Analysis:")
        high_wind_hours = hourly_weather_df[hourly_weather_df['wind_speed_mph'] > 15]
        print(f"   High wind hours (>15mph): {len(high_wind_hours)} of {len(hourly_weather_df)}")
        
        extreme_wind_hours = hourly_weather_df[hourly_weather_df['wind_speed_mph'] > 20]
        print(f"   Extreme wind hours (>20mph): {len(extreme_wind_hours)}")
        
        print(f"\n📊 Sample hourly data (game time hours):")
        game_time_hours = hourly_weather_df[hourly_weather_df['hour'].isin([10, 13, 16, 20])]
        display(game_time_hours[['game_date', 'team', 'hour', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'precip_in', 'condition']].head(20))
    else:
        print("\n⚠️ No hourly weather data fetched")
        hourly_weather_df = pd.DataFrame()

In [0]:
# Calculate fantasy impact for each hour (evolving conditions)

if 'hourly_weather_df' in locals() and len(hourly_weather_df) > 0:
    print("\n" + "="*80)
    print("🎮 Calculating Hourly Fantasy Weather Impact")
    print("="*80)
    
    df = hourly_weather_df.copy()
    
    # Wind impact (per hour)
    df['wind_impact_score'] = df.apply(lambda row: 
        0 if row['is_dome'] else
        1 if row['wind_speed_mph'] < 10 else
        2 if row['wind_speed_mph'] < 15 else
        3 if row['wind_speed_mph'] < 20 else
        4,
        axis=1
    )
    
    # Gust impact (sudden gusts more impactful)
    df['gust_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['wind_gust_mph'] < 15 else
        2 if row['wind_gust_mph'] < 20 else
        3 if row['wind_gust_mph'] < 25 else
        4,
        axis=1
    )
    
    # Cold impact
    df['cold_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] or row['temp_f'] > 50 else
        1 if row['temp_f'] >= 40 else
        2 if row['temp_f'] >= 32 else
        3,
        axis=1
    )
    
    # Precipitation impact (hourly)
    df['precip_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] or row['precip_in'] < 0.01 else
        1 if row['precip_in'] < 0.05 else
        2 if row['precip_in'] < 0.1 else
        3,
        axis=1
    )
    
    # Overall impact (max of all scores)
    df['overall_weather_impact'] = df[['wind_impact_score', 'gust_impact_score', 'cold_impact_score', 'precip_impact_score']].max(axis=1)
    
    # Position adjustments (hourly)
    df['qb_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        -1 * (row['wind_impact_score'] * 5 + row['cold_impact_score'] * 3 + row['precip_impact_score'] * 3),
        axis=1
    )
    
    df['rb_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        (row['cold_impact_score'] * 3 + row['precip_impact_score'] * 4),
        axis=1
    )
    
    df['wr_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        -1 * (row['wind_impact_score'] * 6 + row['cold_impact_score'] * 2 + row['precip_impact_score'] * 3),
        axis=1
    )
    
    df['k_adjustment'] = df.apply(lambda row:
        0 if row['is_dome'] else
        -1 * (row['wind_impact_score'] * 8 + row['gust_impact_score'] * 6 + row['cold_impact_score'] * 2),
        axis=1
    )
    
    hourly_impact_df = df
    
    print(f"\n✅ Calculated impact scores for {len(hourly_impact_df)} hourly records")
    
    # Show worst weather hours
    worst_hours = hourly_impact_df[hourly_impact_df['overall_weather_impact'] >= 3].sort_values('overall_weather_impact', ascending=False)
    
    if len(worst_hours) > 0:
        print(f"\n⚠️ High-Impact Weather Hours ({len(worst_hours)} hours with impact >= 3):")
        print(f"\n📊 Worst conditions by hour:")
        display(worst_hours[['game_date', 'team', 'hour', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'overall_weather_impact', 'qb_adjustment', 'k_adjustment']].head(10))

else:
    print("\n⚠️ No hourly weather data to calculate impact")

In [0]:
# Transform hourly weather to Spark DataFrame

if 'hourly_impact_df' in locals() and len(hourly_impact_df) > 0:
    print("\n" + "="*80)
    print("📦 Transforming Hourly Weather to Spark DataFrame")
    print("="*80)
    
    rows = []
    for idx, row in hourly_impact_df.iterrows():
        spark_row = Row(
            event_id=str(row['event_id']),
            season=int(row['season']),
            week=int(row['week']),
            game_date=str(row['game_date']),
            hour=int(row['hour']),
            team=str(row['team']),
            city=str(row['city']),
            stadium=str(row['stadium']),
            is_dome=bool(row['is_dome']),
            # Temperature
            temp_f=float(row['temp_f']) if pd.notna(row['temp_f']) else None,
            temp_c=float(row['temp_c']) if pd.notna(row['temp_c']) else None,
            feels_like_f=float(row['feels_like_f']) if pd.notna(row['feels_like_f']) else None,
            feels_like_c=float(row['feels_like_c']) if pd.notna(row['feels_like_c']) else None,
            # Wind
            wind_speed_mph=float(row['wind_speed_mph']) if pd.notna(row['wind_speed_mph']) else None,
            wind_speed_kmph=float(row['wind_speed_kmph']) if pd.notna(row['wind_speed_kmph']) else None,
            wind_gust_mph=float(row['wind_gust_mph']) if pd.notna(row['wind_gust_mph']) else None,
            wind_gust_kmph=float(row['wind_gust_kmph']) if pd.notna(row['wind_gust_kmph']) else None,
            wind_dir=str(row['wind_dir']) if pd.notna(row['wind_dir']) and row['wind_dir'] else None,
            wind_degree=int(row['wind_degree']) if pd.notna(row['wind_degree']) else None,
            # Precipitation
            precip_in=float(row['precip_in']) if pd.notna(row['precip_in']) else None,
            precip_mm=float(row['precip_mm']) if pd.notna(row['precip_mm']) else None,
            humidity=int(row['humidity']) if pd.notna(row['humidity']) else None,
            chance_of_rain=int(row['chance_of_rain']) if pd.notna(row['chance_of_rain']) else None,
            chance_of_snow=int(row['chance_of_snow']) if pd.notna(row['chance_of_snow']) else None,
            # Conditions
            condition=str(row['condition']) if pd.notna(row['condition']) and row['condition'] else None,
            condition_code=int(row['condition_code']) if pd.notna(row['condition_code']) else None,
            cloud_cover=int(row['cloud_cover']) if pd.notna(row['cloud_cover']) else None,
            visibility_miles=float(row['visibility_miles']) if pd.notna(row['visibility_miles']) else None,
            # Impact scores
            wind_impact_score=int(row['wind_impact_score']),
            gust_impact_score=int(row['gust_impact_score']),
            cold_impact_score=int(row['cold_impact_score']),
            precip_impact_score=int(row['precip_impact_score']),
            overall_weather_impact=int(row['overall_weather_impact']),
            # Position adjustments
            qb_adjustment=int(row['qb_adjustment']),
            rb_adjustment=int(row['rb_adjustment']),
            wr_adjustment=int(row['wr_adjustment']),
            k_adjustment=int(row['k_adjustment']),
            # Raw data
            raw_data=str(row['raw_data']) if pd.notna(row['raw_data']) else None
        )
        rows.append(spark_row)
    
    hourly_weather_spark_df = spark.createDataFrame(rows)
    print(f"\n✅ Created Spark DataFrame with {hourly_weather_spark_df.count()} hourly records")
    print(f"\n📊 Sample transformed data:")
    display(hourly_weather_spark_df.select('game_date', 'team', 'hour', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'overall_weather_impact').limit(10))
    
else:
    print("\n⚠️ No hourly_impact_df to transform")

In [0]:
# Write hourly weather data to bronze_nfl_weather_hourly table

if 'hourly_weather_spark_df' in locals():
    print("\n" + "="*80)
    print("💾 Writing Hourly Weather Data to bronze_nfl_weather_hourly")
    print("="*80)
    
    bronze_hourly = hourly_weather_spark_df.withColumn("ingested_at", F.current_timestamp())
    bronze_hourly = bronze_hourly.withColumn("source", F.lit("worldweatheronline"))
    
    # Create temp view for merge
    bronze_hourly.createOrReplaceTempView("worldweatheronline_hourly_updates")
    
    # Create hourly weather table if not exists
    spark.sql("""
        CREATE TABLE IF NOT EXISTS main.fantasai.bronze_nfl_weather_hourly (
            event_id STRING,
            season INT,
            week INT,
            game_date STRING,
            hour INT,
            team STRING,
            city STRING,
            stadium STRING,
            is_dome BOOLEAN,
            temp_f DOUBLE,
            temp_c DOUBLE,
            feels_like_f DOUBLE,
            feels_like_c DOUBLE,
            wind_speed_mph DOUBLE,
            wind_speed_kmph DOUBLE,
            wind_gust_mph DOUBLE,
            wind_gust_kmph DOUBLE,
            wind_dir STRING,
            wind_degree INT,
            precip_in DOUBLE,
            precip_mm DOUBLE,
            humidity INT,
            chance_of_rain INT,
            chance_of_snow INT,
            condition STRING,
            condition_code INT,
            cloud_cover INT,
            visibility_miles DOUBLE,
            wind_impact_score INT,
            gust_impact_score INT,
            cold_impact_score INT,
            precip_impact_score INT,
            overall_weather_impact INT,
            qb_adjustment INT,
            rb_adjustment INT,
            wr_adjustment INT,
            k_adjustment INT,
            raw_data STRING,
            source STRING,
            ingested_at TIMESTAMP
        )
        USING DELTA
        COMMENT 'Bronze layer: NFL game hourly weather data for pre-game tactical decisions'
    """)
    
    # Perform MERGE to update existing hourly records and insert new ones
    spark.sql("""
        MERGE INTO main.fantasai.bronze_nfl_weather_hourly AS target
        USING worldweatheronline_hourly_updates AS source
        ON target.event_id = source.event_id 
           AND target.game_date = source.game_date
           AND target.hour = source.hour
           AND target.source = source.source
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    
    total_written = bronze_hourly.count()
    print(f"\n✅ Merged {total_written} hourly weather records into bronze_nfl_weather_hourly")
    
    # Show summary by game
    print("\n📊 Breakdown by Game:")
    bronze_hourly.groupBy("game_date", "team").agg(
        F.count("*").alias("hours"),
        F.avg("temp_f").alias("avg_temp_f"),
        F.avg("wind_speed_mph").alias("avg_wind_mph"),
        F.max("wind_gust_mph").alias("max_gust_mph"),
        F.sum(F.when(F.col("overall_weather_impact") >= 3, 1).otherwise(0)).alias("high_impact_hours")
    ).orderBy("game_date", "team").show(20, truncate=False)
    
    print("\n🎯 Pre-Game Insights:")
    print("   Use this hourly data for:")
    print("   1. Identifying wind trends (increasing/decreasing through day)")
    print("   2. Precise game-time conditions (1pm vs 4pm kickoffs)")
    print("   3. Last-minute sit/start decisions based on weather changes")
    print("   4. Kicker accuracy adjustments for specific game time wind speeds")
    
else:
    print("\n⚠️ No hourly weather data to write")

In [0]:
%sql
-- Verify hourly weather data for upcoming games
SELECT 
    game_date,
    team,
    hour,
    temp_f,
    feels_like_f,
    wind_speed_mph,
    wind_gust_mph,
    wind_dir,
    precip_in,
    condition,
    overall_weather_impact,
    qb_adjustment,
    k_adjustment
FROM main.fantasai.bronze_nfl_weather_hourly
WHERE source = 'worldweatheronline'
  AND game_date >= CURRENT_DATE()
ORDER BY game_date, team, hour
LIMIT 50

In [0]:
%sql
-- Example: Analyze hourly wind trends for afternoon games
-- Helps identify if conditions are getting better or worse

WITH game_kickoffs AS (
  SELECT DISTINCT game_date, team
  FROM main.fantasai.bronze_nfl_weather_hourly
  WHERE game_date >= CURRENT_DATE()
    AND source = 'worldweatheronline'
),
hourly_trends AS (
  SELECT 
    h.game_date,
    h.team,
    h.hour,
    h.wind_speed_mph,
    h.wind_gust_mph,
    h.temp_f,
    h.overall_weather_impact,
    h.qb_adjustment,
    h.k_adjustment,
    -- Compare to morning baseline (10am)
    FIRST_VALUE(h.wind_speed_mph) OVER (PARTITION BY h.game_date, h.team ORDER BY h.hour) as morning_wind,
    -- Trend from previous hour
    LAG(h.wind_speed_mph, 1) OVER (PARTITION BY h.game_date, h.team ORDER BY h.hour) as prev_hour_wind
  FROM main.fantasai.bronze_nfl_weather_hourly h
  INNER JOIN game_kickoffs g ON h.game_date = g.game_date AND h.team = g.team
  WHERE h.source = 'worldweatheronline'
    AND h.hour BETWEEN 10 AND 20  -- Game day hours
)
SELECT 
  game_date,
  team,
  hour,
  wind_speed_mph,
  wind_gust_mph,
  ROUND(wind_speed_mph - morning_wind, 1) as wind_change_from_morning,
  ROUND(wind_speed_mph - COALESCE(prev_hour_wind, wind_speed_mph), 1) as wind_change_last_hour,
  temp_f,
  overall_weather_impact,
  qb_adjustment,
  k_adjustment,
  CASE 
    WHEN hour = 13 THEN '🕑 1pm Kickoff'
    WHEN hour = 16 THEN '🕔 4pm Kickoff'
    WHEN hour = 20 THEN '🌙 8pm Kickoff'
    ELSE ''
  END as kickoff_note
FROM hourly_trends
WHERE hour IN (10, 13, 16, 20)  -- Key game times
ORDER BY game_date, team, hour

## WorldWeatherOnline: Pre-Game Hourly Weather Strategy

### 🎯 Use Case: Tactical Pre-Game Decisions

This notebook fetches **hour-by-hour weather** for upcoming games to help with **last-minute lineup decisions** based on evolving conditions.

### 📅 Recommended Schedule

**Friday Morning (8am)**
- Run for Sunday games
- Get initial hourly trends
- Identify potential weather concerns
- Start monitoring wind patterns

**Saturday Morning (8am)**
- Run again for Sunday games
- Updated hourly forecast
- Compare to Friday's forecast
- Adjust sit/start decisions

**Sunday Morning (10am)**
- Final run before 1pm/4pm games
- Most accurate game-time conditions
- Last-minute kicker/QB adjustments
- Confirm wind speeds at kickoff time

### 🌬️ What to Look For

**Wind Trends**
- Is wind increasing or decreasing through the day?
- Will 1pm game have better/worse conditions than 4pm?
- Are gusts subsiding by kickoff?

**Game-Time Precision**
- 1pm kickoff: Check 13:00 hour
- 4pm kickoff: Check 16:00 hour
- 8pm kickoff: Check 20:00 hour

**Position-Specific Decisions**
- **QB**: If wind >15mph at game time, downgrade
- **Kicker**: If wind >12mph or gusts >20mph, avoid
- **WR/TE**: Deep threats suffer most in wind
- **RB**: Benefits from weather-induced run-heavy scripts

### 📊 Data Tables

**main.fantasai.bronze_nfl_weather_hourly**
- Hour-by-hour weather for upcoming games
- 24 hourly records per game
- Wind speed, gusts, temperature, precipitation
- Fantasy impact scores per hour

**main.fantasai.bronze_nfl_weather**
- Historical daily weather (2016-2025)
- Use for modeling and backtesting
- From WeatherAPI.com (2,639 games)

### 🔢 API Limits

**Free Tier: 500 calls/day**
- Each game = ~2 API calls (one for each date check)
- Can handle ~16 games per run
- Perfect for weekly Sunday slate

**Typical Usage:**
- Friday: ~10-12 games = 20-24 calls
- Saturday: Same games = 20-24 calls
- Sunday: Same games = 20-24 calls
- **Total weekend: 60-72 calls** (well under limit)

### ⚡ Best Practices

1. **Run early in the day** - Get ahead of lineup decisions
2. **Focus on outdoor games** - Dome games don't need hourly updates
3. **Track wind trends** - Is it getting better or worse?
4. **Compare forecasts** - How has the forecast changed since yesterday?
5. **Game time matters** - 1pm vs 4pm can have very different conditions

In [0]:
# Verify hourly weather data coverage for upcoming games
print("\n====================================================")
print("⏱️ WorldWeatherOnline Pre-Game Weather Summary")
print("====================================================\n")

hourly_summary = spark.sql("""
SELECT 
  game_date,
  COUNT(DISTINCT team) as teams_with_weather,
  COUNT(DISTINCT hour) as hours_covered,
  MIN(hour) as first_hour,
  MAX(hour) as last_hour,
  ROUND(AVG(wind_speed_mph), 1) as avg_wind,
  ROUND(AVG(temp_f), 1) as avg_temp,
  COUNT(*) as total_hourly_records
FROM main.fantasai.bronze_nfl_weather_hourly
WHERE source = 'worldweatheronline'
  AND game_date >= CURRENT_DATE()
GROUP BY game_date
ORDER BY game_date
""")

print("\n⏱️ Hourly Weather by Game Date:")
display(hourly_summary)

high_impact_hours = spark.sql("""
SELECT 
  game_date,
  team,
  hour,
  temp_f,
  wind_speed_mph,
  precip_in,
  overall_weather_impact
FROM main.fantasai.bronze_nfl_weather_hourly
WHERE source = 'worldweatheronline'
  AND game_date >= CURRENT_DATE()
  AND (wind_speed_mph > 15 OR precip_in > 0.1 OR temp_f < 32)
ORDER BY game_date, hour
""")

print("\n🌪️ High-Impact Hours (Wind >15mph, Rain >0.1\", Temp <32°F):")
display(high_impact_hours)

print("\n✅ WorldWeatherOnline pre-game weather fetch complete")
print("\n💡 Run this notebook Friday/Saturday/Sunday morning for game-day updates")